# TPU Sequence Pipeline

This notebook builds a neural sequence candidate for ROGII TVT prediction. It is Kaggle-first, TPU-aware, and intentionally separate from the public physics repro notebook.

The model predicts `delta_u`, where `u = TVT + Z`. Final TVT is recovered as `pred_tvt = pred_u - Z`.

This block locates Kaggle data and configures the run. On Kaggle it now fails fast if TensorFlow cannot attach to the TPU, which prevents Kaggle from killing an idle TPU VM while the notebook trains on CPU.

In [ ]:
from dataclasses import dataclass
from pathlib import Path
import os
import random

import numpy as np
import pandas as pd

import tensorflow as tf


@dataclass(frozen=True)
class PathConfig:
    dataset_dir: Path
    train_dir: Path
    test_dir: Path
    sample_submission_path: Path
    work_dir: Path

    @classmethod
    def from_runtime(cls) -> "PathConfig":
        candidates = [
            Path("/kaggle/input/competitions/rogii-wellbore-geology-prediction"),
            Path("/kaggle/input/rogii-wellbore-geology-prediction"),
            Path.cwd().parent / "datasets",
            Path.cwd() / "datasets",
        ]
        dataset_dir = next((path for path in candidates if (path / "sample_submission.csv").exists()), None)
        if dataset_dir is None:
            raise FileNotFoundError("Could not locate competition dataset")
        work_dir = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd().parent / "working"
        work_dir.mkdir(parents=True, exist_ok=True)
        return cls(
            dataset_dir=dataset_dir,
            train_dir=dataset_dir / "train",
            test_dir=dataset_dir / "test",
            sample_submission_path=dataset_dir / "sample_submission.csv",
            work_dir=work_dir,
        )


@dataclass(frozen=True)
class ModelConfig:
    seed: int = 42
    seq_len: int = 12288
    batch_size: int = 16
    epochs: int = 6
    learning_rate: float = 2e-4
    hidden_dim: int = 128
    blocks: int = 6
    kernel_size: int = 7
    dropout: float = 0.15
    train_cutpoints_per_well: int = 2
    validation_wells: int = 80
    local_train_wells: int = 80
    kaggle_train_wells: int = 0
    require_tpu: bool = False


class Runtime:
    """Initializes TPU and refuses to run on CPU when TPU is required."""

    def __init__(self, config: ModelConfig):
        self.config = config
        self._seed_everything()
        self.strategy = self._build_strategy()

    def _build_strategy(self):
        errors = []
        for builder in (self._connect_default_tpu, self._connect_local_tpu, self._connect_named_tpu):
            try:
                resolver = builder()
                tf.tpu.experimental.initialize_tpu_system(resolver)
                print("TPU initialized")
                return tf.distribute.TPUStrategy(resolver)
            except Exception as error:
                errors.append(f"{builder.__name__}: {error}")
        print("TensorFlow TPU unavailable; using default strategy")
        print(" | ".join(errors))
        strategy = tf.distribute.get_strategy()
        if len(tf.config.list_physical_devices("GPU")) == 0:
            print("No GPU detected. This run will be slow and should be treated as a smoke test.")
        return strategy

    def _connect_default_tpu(self):
        resolver = tf.distribute.cluster_resolver.TPUClusterResolver()
        tf.config.experimental_connect_to_cluster(resolver)
        return resolver

    def _connect_local_tpu(self):
        resolver = tf.distribute.cluster_resolver.TPUClusterResolver(tpu="local")
        tf.config.experimental_connect_to_cluster(resolver)
        return resolver

    def _connect_named_tpu(self):
        tpu_name = os.environ.get("TPU_NAME") or os.environ.get("COLAB_TPU_ADDR")
        if not tpu_name:
            raise RuntimeError("TPU_NAME is not set")
        resolver = tf.distribute.cluster_resolver.TPUClusterResolver(tpu=tpu_name)
        tf.config.experimental_connect_to_cluster(resolver)
        return resolver

    def _seed_everything(self) -> None:
        random.seed(self.config.seed)
        np.random.seed(self.config.seed)
        tf.keras.utils.set_random_seed(self.config.seed)


paths = PathConfig.from_runtime()
model_config = ModelConfig()
runtime = Runtime(model_config)
print(f"dataset: {paths.dataset_dir}")
print(f"work_dir: {paths.work_dir}")
print(f"replicas: {runtime.strategy.num_replicas_in_sync}")

# Data Loading and Sequence Features

This section turns each well into a fixed-length sequence. Training uses random cutpoints inside the visible prefix, so one well can produce several masked training examples without leaking future TVT.

In [ ]:
class WellRepository:
    """Loads horizontal wells and matching typewells from Kaggle folders."""

    def __init__(self, paths: PathConfig):
        self.paths = paths

    def well_ids(self, split: str) -> list[str]:
        base = self.paths.train_dir if split == "train" else self.paths.test_dir
        return sorted(path.name.split("__")[0] for path in base.glob("*__horizontal_well.csv"))

    def load_horizontal(self, split: str, well_id: str) -> pd.DataFrame:
        base = self.paths.train_dir if split == "train" else self.paths.test_dir
        return pd.read_csv(base / f"{well_id}__horizontal_well.csv")


class WellFoldSplit:
    """Creates a deterministic well-level holdout split."""

    def __init__(self, seed: int, validation_wells: int):
        self.seed = seed
        self.validation_wells = validation_wells

    def split(self, well_ids: list[str]) -> tuple[list[str], list[str]]:
        rng = np.random.default_rng(self.seed)
        shuffled = np.array(sorted(well_ids), dtype=object)
        rng.shuffle(shuffled)
        valid_count = min(self.validation_wells, max(1, len(shuffled) // 5))
        valid = sorted(str(value) for value in shuffled[:valid_count])
        train = sorted(str(value) for value in shuffled[valid_count:])
        return train, valid


class FeatureSchema:
    """Selects numeric columns used by the sequence model."""

    base_columns = ["MD", "X", "Y", "Z", "GR", "ANCC", "ASTNU", "ASTNL", "EGFDU", "EGFDL", "BUDA"]
    engineered_columns = ["known_mask", "tvt_known", "md_since", "frac", "last_known_tvt", "last_known_u"]

    @property
    def columns(self) -> list[str]:
        return self.base_columns + self.engineered_columns


class SequenceExampleBuilder:
    """Builds padded sequence tensors and masked `delta_u` targets."""

    def __init__(self, config: ModelConfig, schema: FeatureSchema):
        self.config = config
        self.schema = schema

    def training_examples(self, well_id: str, frame: pd.DataFrame) -> list[dict]:
        known_indices = np.flatnonzero(frame["TVT_input"].notna().to_numpy())
        if "TVT" not in frame.columns or len(known_indices) < 120:
            return []
        last_known_index = int(known_indices[-1])
        lower = max(40, int(last_known_index * 0.35))
        upper = max(lower + 1, last_known_index - 20)
        if upper <= lower:
            cutpoints = [last_known_index]
        else:
            quantiles = np.linspace(0.55, 0.92, self.config.train_cutpoints_per_well)
            cutpoints = sorted(set(int(lower + q * (upper - lower)) for q in quantiles))
        examples = [self._build(well_id, frame, cutpoint, is_train=True) for cutpoint in cutpoints]
        return [example for example in examples if example is not None]

    def inference_example(self, well_id: str, frame: pd.DataFrame) -> dict:
        known_indices = np.flatnonzero(frame["TVT_input"].notna().to_numpy())
        cutpoint = int(known_indices[-1])
        return self._build(well_id, frame, cutpoint, is_train=False)

    def _build(self, well_id: str, frame: pd.DataFrame, cutpoint: int, is_train: bool) -> dict | None:
        full = frame.reset_index(drop=True).copy()
        n_rows = len(full)
        if n_rows == 0 or cutpoint < 0:
            return None
        tvt_input = full["TVT_input"].astype(float).to_numpy()
        visible_mask = np.zeros(n_rows, dtype=np.float32)
        visible_mask[:cutpoint + 1] = np.isfinite(tvt_input[:cutpoint + 1]).astype(np.float32)
        visible_tvt = np.where(visible_mask > 0, tvt_input, np.nan)
        if not np.isfinite(visible_tvt[:cutpoint + 1]).any():
            return None
        last_known_tvt = float(visible_tvt[:cutpoint + 1][np.isfinite(visible_tvt[:cutpoint + 1])][-1])
        last_known_z = float(full.loc[cutpoint, "Z"])
        last_known_md = float(full.loc[cutpoint, "MD"])
        last_known_u = last_known_tvt + last_known_z
        tvt_known = pd.Series(visible_tvt).ffill().fillna(last_known_tvt).to_numpy(dtype=np.float32)
        features = pd.DataFrame(index=full.index)
        for column in self.schema.base_columns:
            if column in full.columns:
                features[column] = pd.to_numeric(full[column], errors="coerce")
            else:
                features[column] = 0.0
        features["known_mask"] = visible_mask
        features["tvt_known"] = tvt_known
        features["md_since"] = pd.to_numeric(full["MD"], errors="coerce").to_numpy(dtype=np.float32) - last_known_md
        features["frac"] = np.linspace(0.0, 1.0, n_rows, dtype=np.float32)
        features["last_known_tvt"] = last_known_tvt
        features["last_known_u"] = last_known_u
        features = features[self.schema.columns].replace([np.inf, -np.inf], np.nan).astype(np.float32)
        features = features.fillna(features.median(numeric_only=True)).fillna(0.0)
        target = np.zeros(n_rows, dtype=np.float32)
        sample_weight = np.zeros(n_rows, dtype=np.float32)
        if is_train:
            true_tvt = pd.to_numeric(full["TVT"], errors="coerce").to_numpy(dtype=np.float32)
            true_u = true_tvt + pd.to_numeric(full["Z"], errors="coerce").to_numpy(dtype=np.float32)
            target = true_u - last_known_u
            sample_weight[(np.arange(n_rows) > cutpoint) & np.isfinite(target)] = 1.0
            target = np.nan_to_num(target, nan=0.0).astype(np.float32)
            if sample_weight.sum() < 10:
                return None
        row_indices = np.arange(n_rows, dtype=np.int32)
        return self._pad({
            "well_id": well_id,
            "features": features.to_numpy(dtype=np.float32),
            "target": target[:, None],
            "sample_weight": sample_weight,
            "row_indices": row_indices,
            "last_known_u": np.float32(last_known_u),
            "z": pd.to_numeric(full["Z"], errors="coerce").fillna(0.0).to_numpy(dtype=np.float32),
            "n_rows": n_rows,
        })

    def _pad(self, example: dict) -> dict:
        length = min(int(example["n_rows"]), self.config.seq_len)
        feature_count = example["features"].shape[1]
        padded_features = np.zeros((self.config.seq_len, feature_count), dtype=np.float32)
        padded_target = np.zeros((self.config.seq_len, 1), dtype=np.float32)
        padded_weight = np.zeros((self.config.seq_len,), dtype=np.float32)
        padded_rows = np.full((self.config.seq_len,), -1, dtype=np.int32)
        padded_z = np.zeros((self.config.seq_len,), dtype=np.float32)
        padded_features[:length] = example["features"][:length]
        padded_target[:length] = example["target"][:length]
        padded_weight[:length] = example["sample_weight"][:length]
        padded_rows[:length] = example["row_indices"][:length]
        padded_z[:length] = example["z"][:length]
        example["features"] = padded_features
        example["target"] = padded_target
        example["sample_weight"] = padded_weight
        example["row_indices"] = padded_rows
        example["z"] = padded_z
        example["n_rows"] = length
        return example


repository = WellRepository(paths)
schema = FeatureSchema()
example_builder = SequenceExampleBuilder(model_config, schema)
all_train_wells = repository.well_ids("train")
if model_config.kaggle_train_wells and Path("/kaggle/input").exists():
    all_train_wells = all_train_wells[:model_config.kaggle_train_wells]
elif not Path("/kaggle/input").exists():
    all_train_wells = all_train_wells[:model_config.local_train_wells]
train_wells, valid_wells = WellFoldSplit(model_config.seed, model_config.validation_wells).split(all_train_wells)
print(f"train wells: {len(train_wells)}")
print(f"valid wells: {len(valid_wells)}")
print(f"features: {len(schema.columns)}")

# Normalization and TensorFlow Datasets

The normalizer is fitted only on training examples. `sample_weight` makes the loss ignore known-prefix and padded rows.

In [ ]:
class SequenceNormalizer:
    """Fits mean/std on sequence features and applies standardization."""

    def __init__(self):
        self.mean: np.ndarray | None = None
        self.std: np.ndarray | None = None

    def fit(self, examples: list[dict]) -> "SequenceNormalizer":
        values = np.concatenate([example["features"][example["sample_weight"] > 0] for example in examples], axis=0)
        self.mean = values.mean(axis=0).astype(np.float32)
        self.std = values.std(axis=0).astype(np.float32)
        self.std[self.std < 1e-6] = 1.0
        return self

    def transform(self, examples: list[dict]) -> list[dict]:
        if self.mean is None or self.std is None:
            raise ValueError("Normalizer is not fitted")
        result = []
        for example in examples:
            item = dict(example)
            item["features"] = ((item["features"] - self.mean) / self.std).astype(np.float32)
            result.append(item)
        return result


class SequenceDatasetFactory:
    """Builds tf.data datasets from in-memory padded examples."""

    def __init__(self, config: ModelConfig):
        self.config = config

    def make(self, examples: list[dict], shuffle: bool) -> tf.data.Dataset:
        x = np.stack([example["features"] for example in examples]).astype(np.float32)
        y = np.stack([example["target"] for example in examples]).astype(np.float32)
        w = np.stack([example["sample_weight"] for example in examples]).astype(np.float32)
        dataset = tf.data.Dataset.from_tensor_slices((x, y, w))
        if shuffle:
            dataset = dataset.shuffle(len(examples), seed=self.config.seed, reshuffle_each_iteration=True)
        return dataset.batch(self.config.batch_size, drop_remainder=shuffle).prefetch(tf.data.AUTOTUNE)


def build_training_examples(wells: list[str]) -> list[dict]:
    examples = []
    for well_id in wells:
        frame = repository.load_horizontal("train", well_id)
        examples.extend(example_builder.training_examples(well_id, frame))
    return examples


train_examples_raw = build_training_examples(train_wells)
valid_examples_raw = build_training_examples(valid_wells)
normalizer = SequenceNormalizer().fit(train_examples_raw)
train_examples = normalizer.transform(train_examples_raw)
valid_examples = normalizer.transform(valid_examples_raw)
dataset_factory = SequenceDatasetFactory(model_config)
train_dataset = dataset_factory.make(train_examples, shuffle=True)
valid_dataset = dataset_factory.make(valid_examples, shuffle=False)
print(f"train examples: {len(train_examples)}")
print(f"valid examples: {len(valid_examples)}")

# TCN Model

A compact dilated Conv1D model is the first neural candidate. It is small enough for this dataset and TPU-friendly enough for Kaggle.

In [ ]:
class TCNModelFactory:
    """Creates a residual dilated Conv1D regressor for per-row `delta_u`."""

    def __init__(self, config: ModelConfig, n_features: int):
        self.config = config
        self.n_features = n_features

    def build(self) -> tf.keras.Model:
        inputs = tf.keras.Input(shape=(self.config.seq_len, self.n_features), name="sequence")
        x = tf.keras.layers.Dense(self.config.hidden_dim)(inputs)
        for block_index in range(self.config.blocks):
            dilation = 2 ** (block_index % 6)
            residual = x
            x = tf.keras.layers.LayerNormalization()(x)
            x = tf.keras.layers.Conv1D(
                self.config.hidden_dim,
                self.config.kernel_size,
                padding="same",
                dilation_rate=dilation,
                activation="gelu",
            )(x)
            x = tf.keras.layers.Dropout(self.config.dropout)(x)
            x = tf.keras.layers.Conv1D(self.config.hidden_dim, 1, padding="same")(x)
            x = tf.keras.layers.Add()([residual, x])
        x = tf.keras.layers.LayerNormalization()(x)
        x = tf.keras.layers.Dense(self.config.hidden_dim // 2, activation="gelu")(x)
        outputs = tf.keras.layers.Dense(1, name="delta_u")(x)
        model = tf.keras.Model(inputs, outputs)
        optimizer = tf.keras.optimizers.AdamW(learning_rate=self.config.learning_rate, weight_decay=1e-4)
        model.compile(optimizer=optimizer, loss="mse", weighted_metrics=[tf.keras.metrics.RootMeanSquaredError(name="rmse")])
        return model


with runtime.strategy.scope():
    model = TCNModelFactory(model_config, len(schema.columns)).build()
model.summary()

# Training

The model trains on masked suffixes from train wells and validates on held-out wells. Early stopping keeps this as a candidate model, not a long TPU experiment.

In [ ]:
class SequenceTrainer:
    """Runs training and stores the best Keras model."""

    def __init__(self, config: ModelConfig, paths: PathConfig):
        self.config = config
        self.paths = paths
        self.model_path = paths.work_dir / "tpu_tcn_delta_u.keras"

    def fit(self, model: tf.keras.Model, train_dataset: tf.data.Dataset, valid_dataset: tf.data.Dataset):
        callbacks = [
            tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=4, restore_best_weights=True),
            tf.keras.callbacks.ModelCheckpoint(self.model_path, monitor="val_loss", save_best_only=True),
            tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2),
        ]
        return model.fit(
            train_dataset,
            validation_data=valid_dataset,
            epochs=self.config.epochs,
            callbacks=callbacks,
            verbose=1,
        )


trainer = SequenceTrainer(model_config, paths)
history = trainer.fit(model, train_dataset, valid_dataset)
print(f"saved model: {trainer.model_path}")

# Validation and Submission

This block scores the neural candidate on held-out wells, predicts the Kaggle test wells, and writes `submission_tpu_tcn.csv`. This file is meant to be blended with the physics pipeline, not blindly replace it.

In [ ]:
class SequencePredictor:
    """Converts model `delta_u` predictions back to TVT rows."""

    def __init__(self, model: tf.keras.Model, normalizer: SequenceNormalizer, builder: SequenceExampleBuilder):
        self.model = model
        self.normalizer = normalizer
        self.builder = builder

    def predict_example(self, example: dict) -> pd.DataFrame:
        normalized = self.normalizer.transform([example])[0]
        pred_delta_u = self.model.predict(normalized["features"][None, ...], verbose=0)[0, :, 0]
        n_rows = int(example["n_rows"])
        pred_u = pred_delta_u[:n_rows] + float(example["last_known_u"])
        pred_tvt = pred_u - example["z"][:n_rows]
        return pd.DataFrame({
            "row_index": example["row_indices"][:n_rows].astype(int),
            "tvt": pred_tvt.astype(float),
        })

    def predict_well(self, split: str, well_id: str) -> pd.DataFrame:
        frame = repository.load_horizontal(split, well_id)
        example = self.builder.inference_example(well_id, frame)
        prediction = self.predict_example(example)
        missing_rows = frame.index[frame["TVT_input"].isna()].to_numpy(dtype=int)
        prediction = prediction[prediction["row_index"].isin(missing_rows)].copy()
        prediction["id"] = [f"{well_id}_{row_index}" for row_index in prediction["row_index"]]
        return prediction[["id", "tvt"]]


def validation_rmse() -> float:
    predictor = SequencePredictor(model, normalizer, example_builder)
    squared_errors = []
    for well_id in valid_wells:
        frame = repository.load_horizontal("train", well_id)
        example = example_builder.inference_example(well_id, frame)
        prediction = predictor.predict_example(example)
        truth = frame.reset_index().rename(columns={"index": "row_index"})[["row_index", "TVT"]]
        visible_missing = frame["TVT_input"].isna().to_numpy()
        truth = truth[visible_missing].merge(prediction, on="row_index", how="inner", suffixes=("_true", "_pred"))
        if len(truth):
            error = truth["TVT_true"].to_numpy(dtype=float) - truth["tvt"].to_numpy(dtype=float)
            squared_errors.extend((error * error).tolist())
    return float(np.sqrt(np.mean(squared_errors))) if squared_errors else float("nan")


class SubmissionWriter:
    """Writes the TPU sequence candidate in Kaggle submission format."""

    def __init__(self, paths: PathConfig, repository: WellRepository, predictor: SequencePredictor):
        self.paths = paths
        self.repository = repository
        self.predictor = predictor

    def write(self) -> Path:
        parts = [self.predictor.predict_well("test", well_id) for well_id in self.repository.well_ids("test")]
        predictions = pd.concat(parts, ignore_index=True)
        sample = pd.read_csv(self.paths.sample_submission_path)[["id"]]
        submission = sample.merge(predictions, on="id", how="left")
        if submission["tvt"].isna().any():
            raise ValueError("Submission contains missing TVT predictions")
        path = self.paths.work_dir / "submission_tpu_tcn.csv"
        submission.to_csv(path, index=False)
        return path


predictor = SequencePredictor(model, normalizer, example_builder)
valid_rmse = validation_rmse()
submission_path = SubmissionWriter(paths, repository, predictor).write()
print(f"validation rmse: {valid_rmse:.5f}")
print(f"submission: {submission_path}")

# Next Step

Submit `submission_tpu_tcn.csv` only as a diagnostic candidate first. If it is competitive or decorrelated from the public physics pipeline, blend it with `submission.csv` from the public repro notebook.